# ν-Support Vector Regression (ν-SVR)

## What is Support Vector Machine?
Support Vector Machine (SVM) is an Artificial Intelligence technique that
uses a **kernel function** to map inseparable input data to a higher
dimensional hyperspace where it becomes linearly separable.

For **regression problems** (predicting continuous values like stock returns),
it is called **Support Vector Regression (SVR)**.

## Why ν-SVR instead of standard SVR?
Standard SVR has a parameter **ε** (epsilon) which:
- Controls the tube radius around the regression line
- Is very sensitive to prediction performance
- Difficult to set manually → causes uncertainty

Schölkopf et al. proposed **ν-SVR** which:
- Replaces ε with a new parameter **ν**
- ν controls upper bound on fraction of errors
- ν controls lower bound on fraction of support vectors
- Much easier to set with clearer meaning
- ν ∈ (0, 1] — easier to understand than ε

## RBF Kernel
We use the **Gaussian Radial Basis Function (RBF)** kernel:

    k(x1, x2) = exp(-λ||x - y||²)

Where:
- λ (lambda) is the kernel parameter
- It maps data into infinite dimensional space
- Allows fitting complex nonlinear relationships

## Three Parameters to Tune
| Parameter | Meaning | Paper Value |
|-----------|---------|-------------|
| C | Penalty constant — controls margin vs error tradeoff | 2 |
| ν | Controls support vectors and errors | 0.5 |
| λ | RBF kernel width — most sensitive parameter | 0.001 |

## Grid Search
We find optimal parameters using **exhaustive grid search**:
- Try all combinations of C, ν, λ
- Pick combination with best validation R²
- Paper range: C ∈ {0.1,1,...,1000}, λ ∈ {1e-7,...,1e6}, ν ∈ {0.1,...,0.5}
- We use a smaller grid to save computation time

## Why SVR for Stock Prediction?
- Handles **nonlinear relationships** unlike FFFFM
- Uses **structural risk minimization** (better generalization)
- Fewer parameters than deep learning
- Training is faster than neural networks for medium-sized data
- SVM has shown to outperform traditional ANNs in many regression tasks

## Evaluation Metric — R²
Same as FFFFM — R² (coefficient of determination)
Higher R² = better prediction of stock returns

In [9]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.svm import NuSVR
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── ENVIRONMENT SETUP ──────────────────────────────────────
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/portfolio_project'
else:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..','data','portfolio_project'))

print(f"Using: {BASE_DIR}")

# ── LOAD DATA ──────────────────────────────────────────────
train = pd.read_csv(os.path.join(BASE_DIR, 'train_data.csv'),
                    index_col='Date', parse_dates=True)
valid = pd.read_csv(os.path.join(BASE_DIR, 'valid_data.csv'),
                    index_col='Date', parse_dates=True)

print(f"Train: {train.shape}")
print(f"Valid: {valid.shape}")

# ── FEATURES & TARGETS ─────────────────────────────────────
feature_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

X_train_raw = train[feature_cols].values
X_valid_raw = valid[feature_cols].values

targets = {
    'MSFT': (train['MSFT_excess'].values, valid['MSFT_excess'].values),
    'AAPL': (train['AAPL_excess'].values, valid['AAPL_excess'].values),
    'SONY': (train['SONY_excess'].values, valid['SONY_excess'].values),
}

Using: /Users/aaronjasonbaptist/Documents/IIT Kharagpur/Academic/Semester 4/Deep Learning /Github/gru-portfolio-management-project-main/data/portfolio_project
Train: (2999, 12)
Valid: (499, 12)


## Why Scale the Data for SVR?

SVR uses distance-based calculations (RBF kernel computes ||x-y||²).
If features have very different scales, larger-valued features will
dominate the distance computation unfairly.

**StandardScaler** transforms each feature to:
- Mean = 0
- Standard deviation = 1

This ensures all 5 factors contribute equally to the kernel computation.
Note: FFFFM (linear regression) does not need scaling, but SVR and GRU do.

In [10]:
# ── SCALE FEATURES ─────────────────────────────────────────
# Fit scaler ONLY on training data
# Then apply same scaler to validation data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_valid = scaler.transform(X_valid_raw)   # ← use same scaler, don't refit

print("Scaling done!")
print(f"X_train mean: {X_train.mean(axis=0).round(4)}")  # should be ~0
print(f"X_train std:  {X_train.std(axis=0).round(4)}")   # should be ~1

Scaling done!
X_train mean: [-0. -0.  0. -0.  0.]
X_train std:  [1. 1. 1. 1. 1.]


## Grid Search Strategy

Full grid search from paper is very slow:
- C: 0.1 to 1000 (many values)
- λ: 1e-7 to 1e6  (many values)  
- ν: 0.1 to 0.5   (5 values)

This would take hours on a single GPU.

Our approach:
1. Start with paper's best values: C=2, ν=0.5, λ=0.001
2. Search a small grid around those values
3. Pick best combination per stock

This is valid because the paper itself confirms these
parameters work well for this dataset.

In [11]:
# ── GRID SEARCH ────────────────────────────────────────────
# Small grid around paper's best values
C_values   = [0.1, 1, 2, 5, 10]
nu_values  = [0.1, 0.2, 0.3, 0.5]
gamma_values = [0.0001, 0.001, 0.01, 0.1]   # gamma = λ in paper

svr_models  = {}
svr_r2      = {}
svr_preds   = {}
svr_params  = {}

print("=" * 50)
print("ν-SVR — Grid Search & Training")
print("=" * 50)

for stock, (y_train, y_valid) in targets.items():
    print(f"\nSearching best params for {stock}...")

    best_r2     = -np.inf
    best_params = {}
    best_model  = None

    for C in C_values:
        for nu in nu_values:
            for gamma in gamma_values:
                try:
                    model = NuSVR(
                        C=C,
                        nu=nu,
                        gamma=gamma,
                        kernel='rbf'
                    )
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_valid)
                    r2 = r2_score(y_valid, y_pred)

                    if r2 > best_r2:
                        best_r2     = r2
                        best_params = {'C': C, 'nu': nu, 'gamma': gamma}
                        best_model  = model
                except:
                    continue

    # Store best
    svr_models[stock] = best_model
    svr_r2[stock]     = best_r2
    svr_params[stock] = best_params
    svr_preds[stock]  = best_model.predict(X_valid)

    print(f"  Best params : {best_params}")
    print(f"  Best R²     : {best_r2:.4f}")

print("\n" + "=" * 50)
print("Summary R² values:")
for stock, r2 in svr_r2.items():
    print(f"  {stock}: {r2:.3f}")

ν-SVR — Grid Search & Training

Searching best params for MSFT...
  Best params : {'C': 2, 'nu': 0.2, 'gamma': 0.0001}
  Best R²     : 0.2899

Searching best params for AAPL...
  Best params : {'C': 2, 'nu': 0.1, 'gamma': 0.0001}
  Best R²     : 0.2338

Searching best params for SONY...
  Best params : {'C': 0.1, 'nu': 0.5, 'gamma': 0.01}
  Best R²     : 0.2165

Summary R² values:
  MSFT: 0.290
  AAPL: 0.234
  SONY: 0.217


In [12]:
print("=" * 50)
print("ν-SVR — 20-Fold Cross Validation & P-values")
print("=" * 50)

X_all = np.vstack([X_train, X_valid])
kf    = KFold(n_splits=20, shuffle=False)

for stock, (y_train, y_valid) in targets.items():
    y_all = np.concatenate([y_train, y_valid])

    # Use best params found
    p = svr_params[stock]
    model = NuSVR(C=p['C'], nu=p['nu'],
                  gamma=p['gamma'], kernel='rbf')

    cv_scores = cross_val_score(model, X_all, y_all,
                                cv=kf, scoring='r2')

    mean_r2 = cv_scores.mean()
    std_r2  = cv_scores.std()

    t_stat, p_value = stats.ttest_1samp(cv_scores, svr_r2[stock])

    print(f"\n{stock}:")
    print(f"  Reported R²  : {svr_r2[stock]:.3f}")
    print(f"  CV Mean R²   : {mean_r2:.3f}")
    print(f"  CV Std R²    : {std_r2:.3f}")
    print(f"  P-value      : {p_value:.4f}")
    print(f"  Significant? : {'Yes' if p_value < 0.05 else 'No'}")

ν-SVR — 20-Fold Cross Validation & P-values

MSFT:
  Reported R²  : 0.290
  CV Mean R²   : 0.436
  CV Std R²    : 0.157
  P-value      : 0.0007
  Significant? : Yes

AAPL:
  Reported R²  : 0.234
  CV Mean R²   : 0.310
  CV Std R²    : 0.141
  P-value      : 0.0295
  Significant? : Yes

SONY:
  Reported R²  : 0.217
  CV Mean R²   : 0.291
  CV Std R²    : 0.097
  P-value      : 0.0036
  Significant? : Yes


In [13]:
# Save SVR predictions on validation set
svr_preds_df = pd.DataFrame(svr_preds, index=valid.index)
svr_preds_df.to_csv(os.path.join(BASE_DIR, 'svr_predictions.csv'))
print("Saved SVR predictions!")
print(svr_preds_df.head())

Saved SVR predictions!
                MSFT      AAPL      SONY
Date                                    
2011-12-05  0.012220  0.018209  0.009859
2011-12-06  0.000557  0.000243 -0.000657
2011-12-07 -0.000838 -0.003046 -0.000102
2011-12-08 -0.017287 -0.016521 -0.026457
2011-12-09  0.016339  0.021307  0.017294
